# Lab 15-EDA: คู่มือ Data Understanding, Visualization, EDA เจาะลึก, Hypothesis Testing และการเลือกโมเดล
> Week 15 | CLO1–CLO4 | ต่อยอดจาก Week 1–14 ทั้งหมด | ใช้ประกอบ Final Project

สมุดนี้เป็น**คู่มือฉบับละเอียด**สำหรับขั้นตอนที่สำคัญที่สุดของ Final Project — สิ่งที่ต้องทำ**หลังจาก**เตรียมข้อมูล (data cleaning) เสร็จแล้ว แต่**ก่อน**เริ่ม fit model จริง เป้าหมายหลักของสมุดนี้คือให้นักศึกษาเข้าใจและฝึกทำ **Data Understanding**, **Data Visualization**, และ **EDA (Exploratory Data Analysis)** อย่างเป็นระบบและละเอียด ซึ่งเป็นขั้นตอนที่มักถูกมองข้ามแต่ส่งผลต่อคุณภาพของทั้ง pipeline มากที่สุด นอกจากนี้เราจะไล่ดูต่อไปถึงการตั้งสมมติฐานจากสิ่งที่เห็นใน EDA แล้วพิสูจน์ด้วย Hypothesis Testing (Week 7), การตั้งโจทย์ (Problem Framing) ให้ตรงกับประเภทข้อมูลที่มี และสุดท้ายคือกรอบการตัดสินใจเลือกโมเดล/วิธีที่เหมาะสมจากทุกวิธีที่เรียนมาในวิชานี้ (Week 8–14) เราจะใช้ **Titanic dataset** เป็นตัวอย่างตล

**:** นายธีรภัทร พิกุลศรี ( 68114540308)

**Dataset:** CASMI26 PubChem Popularity Prior

**Problem Statement:** ---
## Part 5: ตั้งโจทย์ (Problem Framing)

**Part นี้เราจะแปลง insight ทั้งหมดจาก Part 1–4 ให้เป็นโจทย์ Machine Learning ที่ชัดเจน** ขั้นตอนนี้สำคัญมากเพราะการตั้งโจทย์ผิด (เช่น เลือก metric ผิด หรือกำหนด task type ผิด) จะทำให้งานทั้งหมดหลังจากนี้ผิดทิศทาง แม้โมเดลจะ fit ได้ดีแค่ไหนก็ตาม

### 5.1 กำหนด Task Type จากลักษณะของ Target (Y)

| ลักษณะของ Y | Task Type | วิธีที่ใช้ได้ในวิชานี้ |
|---|---|---|
| Continuous (ตัวเลขต่อเนื่อง เช่น ราคา, คะแนน) | **Regression** | SLR/MLR (Week 8–9), Polynomial Regression (Week 10) |
| Binary (2 กลุ่ม เช่น รอด/ไม่รอด, default/ไม่ default) | **Binary Classification** | Logistic, LDA, QDA, Naive Bayes, KNN (Week 11–12) |
| Multi-class (>2 กลุ่มไม่มีลำดับ) | **Multinomial Classification** | Multinomial Lo


In [ ]:
# 
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded successfully")

---
## Part 0: โหลดข้อมูล — CASMI26 PubChem Popularity Prior

**Part นี้เราจะโหลด Titanic dataset** ผ่าน seaborn (ดึงจาก GitHub อัตโนมัติ ไม่ต้อง upload ไฟล์) เพื่อใช้เป็นตัวอย่างตลอดทั้งเล่ม เราเลือก Titanic เพราะ: (1) มีตัวแปร continuous (Age, Fare) และ categorical (Sex, Pclass, Embarked) ปนกัน เหมือน dataset จริงส่วนใหญ่ (2) มี missing values จริงให้ฝึกจัดการ (3) มี target เป็น binary (Survived) ที่ใช้สาธิต classification ได้ครบทุกวิธีที่เรียนมา — **ถ้าใช้กับ Final Project ของตัวเอง ให้แทนที่ cell นี้ด้วยการโหลด dataset ของกลุ่ม แล้วทำตามโครงสร้างเดียวกันได้เลย**

**:**  CASMI26  KaggleHub  regression  popularity


In [ ]:
#  dataset  KaggleHub 
import kagglehub

DATA_DIR = Path(kagglehub.dataset_download("dmitriigluzdov/casmi26-pubchem-popularity-prior"))
#  path  kagglehub  versions/2
if not (DATA_DIR / "pool_lsid.npy").exists():
    DATA_DIR = next(DATA_DIR.glob("versions/*"))

load = lambda name: np.load(DATA_DIR / f"{name}.npy", mmap_mode="r")
lsid = np.asarray(load("pool_lsid"), dtype=np.float32)
lpmid = np.asarray(load("pool_lpmid"), dtype=np.float32)
lpatent = np.asarray(load("pool_lpatent"), dtype=np.float32)
cid = np.asarray(load("pool_cid"), dtype=np.int32)
np_bits = np.asarray(load("pool_np"), dtype=np.uint8)

# popularity  log-counts; features  lpatent  target leakage
df = pd.DataFrame({
    "log_sid": lsid,
    "log_pmid": lpmid,
    "has_pubchem": (cid > 0).astype(np.int8),
    "is_natural_product": (np_bits > 0).astype(np.int8),
    "target_popularity": lsid + lpmid + lpatent,
})
#  sample  notebook  reproducible
if len(df) > 100_000:
    df = df.sample(n=100_000, random_state=42).reset_index(drop=True)

feature_cols = ["log_sid", "log_pmid", "has_pubchem", "is_natural_product"]
target_col = "target_popularity"
print(f"Dataset directory: {DATA_DIR}")
print(f"Data shape: {df.shape}")
print(df.dtypes)
print("Missing values:\n", df.isna().sum())
df.head()

In [ ]:
# Descriptive statistics and visualizations
print(df.describe().T)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df[target_col], bins=40, color="steelblue", edgecolor="white")
axes[0].set(title="Target distribution", xlabel="Popularity prior", ylabel="Count")
axes[1].hist(df["log_sid"], bins=40, alpha=.7, label="log SID")
axes[1].hist(df["log_pmid"], bins=40, alpha=.7, label="log PMID")
axes[1].set(title="Feature distributions"); axes[1].legend()
corr = df[feature_cols + [target_col]].corr()
im = axes[2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[2].set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
axes[2].set_yticks(range(len(corr)), corr.index); axes[2].set_title("Correlation matrix")
fig.colorbar(im, ax=axes[2], fraction=.046)
plt.tight_layout(); plt.show()
print("Correlation with target:")
print(corr[target_col].drop(target_col).sort_values(ascending=False))

---
## Part 2: CLO1 — การวิเคราะห์ด้วยพีชคณิตเชิงเส้น

**Part นี้เราจะสำรวจ "รูปร่าง" ของข้อมูลก่อนวิเคราะห์เชิงลึก** เพื่อรู้ว่ามีปัญหาอะไรซ่อนอยู่บ้าง (missing values, duplicates, ประเภทข้อมูลผิด) ก่อนที่จะเสียเวลาวิเคราะห์ต่อบนข้อมูลที่มีปัญหา **หลักการสำคัญ**: ห้าม fit model ก่อนเข้าใจข้อมูลอย่างถ่องแท้ — เพราะ garbage in, garbage out

ในโครงงานนี้ใช้ PCA และ covariance matrix เพื่อวิเคราะห์ feature matrix ของข้อมูล popularity


In [ ]:
# PCA  covariance matrix
X = df[feature_cols].to_numpy(dtype=float)
y = df[target_col].to_numpy(dtype=float)
X_scaled = StandardScaler().fit_transform(X)
C = np.cov(X_scaled, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(C)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]; eigenvectors = eigenvectors[:, order]
explained = eigenvalues / eigenvalues.sum()
X_pca = X_scaled @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained)+1), explained*100, color="steelblue")
axes[0].plot(range(1, len(explained)+1), np.cumsum(explained)*100, "o-r")
axes[0].set(xlabel="Principal component", ylabel="Explained variance (%)", title="Scree plot")
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, s=5, alpha=.25, cmap="viridis")
axes[1].set(xlabel="PC1", ylabel="PC2", title="PCA projection")
fig.colorbar(sc, ax=axes[1], label="")
plt.tight_layout(); plt.show()
print("Explained variance (%):", np.round(explained*100, 2))
print("PC1 loadings:")
print(pd.Series(eigenvectors[:, 0], index=feature_cols).sort_values(key=np.abs, ascending=False))

**Insight จาก Part 1**: บันทึกสิ่งที่พบเพื่อใช้ตัดสินใจในขั้นตอนถัดไป — เช่น "age missing 19.9% ต้อง impute", "deck missing เกิน 77% ควรตัดทิ้งหรือสร้างเป็น flag แทน", "embarked missing แค่ 0.2% impute ด้วย mode ได้เลย"


---
## Part 2: Data Visualization เจาะลึก

**Part นี้คือหัวใจของสมุดนี้** — เราจะสร้างกราฟหลายมุมมองเพื่อ "เห็น" ข้อมูลก่อนวิเคราะห์เชิงตัวเลข เพราะสมองมนุษย์จับ pattern จากภาพได้เร็วกว่าตาราง (Anscombe's Quartet คือตัวอย่างคลาสสิกที่แสดงว่าข้อมูล 4 ชุดที่มี mean/variance/correlation เหมือนกันทุกประการ สามารถมีรูปร่างต่างกันโดยสิ้นเชิงเมื่อ plot) เราจะทำ 4 ระดับ: **Univariate** (มองทีละตัวแปร) → **Bivariate** (มองความสัมพันธ์ 2 ตัวแปร) → **Target-focused** (มองว่าแต่ละ feature สัมพันธ์กับ target อย่างไร) → **Multivariate** (มองหลายตัวแปรพร้อมกัน)

  popularity  features  target


In [ ]:
# Correlation  bias-variance U-curve
from sklearn.model_selection import train_test_split
X_one = df[["log_sid"]].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X_one, y, test_size=.2, random_state=42)
degrees = range(1, 8); train_mse, test_mse = [], []
for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(Xtr, ytr)
    train_mse.append(mean_squared_error(ytr, model.predict(Xtr)))
    test_mse.append(mean_squared_error(yte, model.predict(Xte)))
plt.figure(figsize=(8, 4)); plt.plot(degrees, train_mse, "o-", label="Train MSE"); plt.plot(degrees, test_mse, "o-", label="Test MSE")
plt.xlabel("Polynomial degree"); plt.ylabel("MSE"); plt.title("Bias-Variance trade-off"); plt.legend(); plt.show()
print("Best test-MSE degree:", degrees[int(np.argmin(test_mse))])
print(df[feature_cols + [target_col]].corr()[target_col].drop(target_col).sort_values(ascending=False))

**Insight จาก Part 2**: จากกราฟทั้งหมด ควรสรุปเป็นข้อความสั้นๆ เช่น "Sex และ Pclass ดูมีผลต่อ Survived ชัดเจนมาก (survival rate ต่างกันมาก), Age มีผลปานกลาง (คนอายุน้อยรอดมากกว่าเล็กน้อย), Fare correlate กับ Pclass สูง (อาจเกิด multicollinearity)" — insight เหล่านี้จะถูกนำไปพิสูจน์อย่างเป็นทางการใน Part 4


---
## Part 5: ตั้งโจทย์ (Problem Framing)

**Part นี้เราจะแปลง insight ทั้งหมดจาก Part 1–4 ให้เป็นโจทย์ Machine Learning ที่ชัดเจน** ขั้นตอนนี้สำคัญมากเพราะการตั้งโจทย์ผิด (เช่น เลือก metric ผิด หรือกำหนด task type ผิด) จะทำให้งานทั้งหมดหลังจากนี้ผิดทิศทาง แม้โมเดลจะ fit ได้ดีแค่ไหนก็ตาม

### 5.1 กำหนด Task Type จากลักษณะของ Target (Y)

| ลักษณะของ Y | Task Type | วิธีที่ใช้ได้ในวิชานี้ |
|---|---|---|
| Continuous (ตัวเลขต่อเนื่อง เช่น ราคา, คะแนน) | **Regression** | SLR/MLR (Week 8–9), Polynomial Regression (Week 10) |
| Binary (2 กลุ่ม เช่น รอด/ไม่รอด, default/ไม่ default) | **Binary Classification** | Logistic, LDA, QDA, Naive Bayes, KNN (Week 11–12) |
| Multi-class (>2 กลุ่มไม่มีลำดับ) | **Multinomial Classification** | Multinomial Logistic, LDA, QDA, Naive Bayes, KNN (Week 11–12) |
| Count (จำนวนนับ 0,1,2,... เช่น จำนวนครั้ง) | **Count Regression** | Poisson Regression / GLM (Week 13) |

**สำหรับตัวอย่างของเรา**: `survived` เป็น binary (0/1) → **Binary Classification**

**:** Regression  target 


In [ ]:
#  train/test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

model1 = LinearRegression().fit(X_train[:, [0]], y_train)
pred1_train = model1.predict(X_train[:, [0]]); pred1_test = model1.predict(X_test[:, [0]])
model2 = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_train, y_train)
pred2_train = model2.predict(X_train); pred2_test = model2.predict(X_test)

def regression_report(name, actual_train, pred_train, actual_test, pred_test):
    return {"Model": name, "Train MSE": mean_squared_error(actual_train, pred_train),
            "Test MSE": mean_squared_error(actual_test, pred_test),
            "Train R2": r2_score(actual_train, pred_train), "Test R2": r2_score(actual_test, pred_test)}
comparison = pd.DataFrame([
    regression_report("Simple Linear Regression", y_train, pred1_train, y_test, pred1_test),
    regression_report("Ridge (all features)", y_train, pred2_train, y_test, pred2_test),
])
print(comparison.to_string(index=False))
print("Simple-model coefficient:", model1.coef_[0], "intercept:", model1.intercept_)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, pred, title in [(ax[0], pred1_test, "Linear Regression "), (ax[1], pred2_test, "Ridge")]:
    a.scatter(y_test, pred, s=5, alpha=.25); lo, hi = y_test.min(), y_test.max(); a.plot([lo,hi],[lo,hi], "r--")
    a.set(xlabel="Actual", ylabel="Predicted", title=title)
plt.tight_layout(); plt.show()

---
## Part 6: เลือกโมเดลและวิธีที่เหมาะสม (Model & Method Selection)

**Part สุดท้ายนี้เราจะใช้กรอบการตัดสินใจที่ครอบคลุมทุกวิธีในวิชานี้** เพื่อเลือกว่าควรลองโมเดลไหนบ้าง แล้ว fit จริง เปรียบเทียบด้วย Cross-Validation (Week 14) และเลือกโมเดลสุดท้ายอย่างมีเหตุผล — ไม่ใช่แค่เลือกตัวที่ accuracy สูงสุดจาก run เดียว


### 6.2 เปรียบเทียบอย่างเป็นธรรมด้วย k-Fold Cross-Validation (Week 14)

การเปรียบเทียบด้วย test set ครั้งเดียว (ด้านบน) อาจได้ผลที่ผันผวนจาก random split — เราต้องใช้ **k-Fold CV** บน training set เพื่อได้ estimate ที่น่าเชื่อถือกว่า ก่อนตัดสินใจเลือกโมเดลสุดท้าย


In [ ]:
# 5-Fold Cross-Validation
#  feature set 
models = {
    "Simple Linear (log SID)": (make_pipeline(StandardScaler(), LinearRegression()), X[:, [0]]),
    "Ridge (all features)": (make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X),
    "Polynomial degree 2": (make_pipeline(PolynomialFeatures(2), Ridge(alpha=1.0)), X[:, [0]]),
}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, (model, features) in models.items():
    scores = -cross_val_score(model, features, y, cv=kfold, scoring="neg_mean_squared_error", n_jobs=-1)
    cv_rows.append({"Model": name, "CV MSE mean": scores.mean(), "CV MSE std": scores.std()})
cv_results = pd.DataFrame(cv_rows).sort_values("CV MSE mean")
print(cv_results.to_string(index=False))
plt.figure(figsize=(9, 4)); plt.bar(cv_results["Model"], cv_results["CV MSE mean"], yerr=cv_results["CV MSE std"], capsize=5, color="steelblue")
plt.ylabel("5-fold CV MSE"); plt.title("Model selection by cross-validation"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()
best_model_name = cv_results.iloc[0]["Model"]
print(f"Best model: {best_model_name}; CV MSE = {cv_results.iloc[0]['CV MSE mean']:.4f}")

### 6.2 เปรียบเทียบอย่างเป็นธรรมด้วย k-Fold Cross-Validation (Week 14)

การเปรียบเทียบด้วย test set ครั้งเดียว (ด้านบน) อาจได้ผลที่ผันผวนจาก random split — เราต้องใช้ **k-Fold CV** บน training set เพื่อได้ estimate ที่น่าเชื่อถือกว่า ก่อนตัดสินใจเลือกโมเดลสุดท้าย

**:** Ridge (all features)  CV MSE  1.3322


---
## สรุปทั้งหมด: จาก EDA สู่ Model Selection

| Part | สิ่งที่ทำ | ผลลัพธ์ที่ได้ |
|------|----------|-------------|
| 1. Data Understanding | ตรวจ shape, dtypes, missing, duplicates, cardinality | รู้ว่าต้อง impute age/embarked, ตัด/flag deck |
| 2. Visualization | Univariate, bivariate, target-focused, multivariate plots | เห็น pattern เบื้องต้นว่า sex, pclass, age น่าจะมีผล |
| 3. EDA เจาะลึก | Outlier (IQR), distribution shape (skew/QQ/Shapiro), VIF | ตัดสินใจ preprocessing strategy อย่างมีหลักฐาน |
| 4. Hypothesis Testing | Chi-square, t-test/Mann-Whitney พิสูจน์สิ่งที่เห็นใน EDA | ยืนยันด้วยตัวเลขว่า feature ไหนมีผลจริง (ไม่ใช่ดูด้วยตา) |
| 5. Problem Framing | กำหนด X, Y, task type, metric, train/test split | โจทย์ชัดเจน: Binary Classification, วัดด้วย AUC+Accuracy |
| 6. Model Selection | Fit 5 methods, CV, One-SE Rule, final test evaluation | เลือกโมเดลสุดท้ายอย่างมีเหตุผล ไม่ใช่สุ่มเลือก |

**นี่คือโครงสร้างเดียวกับที่ Final Project ต้องการ** (CLO1–4) — ต่างกันแค่ว่า Final Project เพิ่ม CLO1 (Linear Algebra: Covariance/PCA) เข้าไปก่อน Part 4 ของสมุดนี้ ดูรายละเอียดได้ที่ `final_project_guidelines.md` และใช้ `lab15_project_template.ipynb` เป็นโครงร่างสำหรับกรอกงานของกลุ่มตัวเอง

**:** target popularity  prior  count data   sample 100,000 

**:**  pc_*  out-of-core  ranking metrics

**:**  KaggleHub
